In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 3.2 Building Tree-Based Models in Practice
- Same `instantiate -> fit -> predict` for all three
- No feature scaling needed!
- Build, evaluate, and compare on student departure data

## Setup

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report)
from sklearn.model_selection import cross_val_score, StratifiedKFold
import time

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries imported successfully!")

## Load and Prepare Data

In [ ]:
train_df = pd.read_csv('../data/training.csv')
test_df = pd.read_csv('../data/testing.csv')

train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

print(f"Training set: {train_df.shape[0]:,} students")
print(f"Testing set: {test_df.shape[0]:,} students")
print(f"Departure rate (Train): {train_df['DEPARTED'].mean():.2%}")

In [ ]:
numeric_features = [
    'HS_GPA', 'HS_MATH_GPA', 'HS_ENGL_GPA',
    'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1', 'UNITS_COMPLETED_2',
    'DFW_UNITS_1', 'DFW_UNITS_2',
    'GPA_1', 'GPA_2',
    'DFW_RATE_1', 'DFW_RATE_2',
    'GRADE_POINTS_1', 'GRADE_POINTS_2'
]
categorical_features = ['RACE_ETHNICITY', 'GENDER', 'FIRST_GEN_STATUS', 'COLLEGE']
target = 'DEPARTED'

train_encoded = pd.get_dummies(train_df[numeric_features + categorical_features],
                               columns=categorical_features, drop_first=True)
test_encoded = pd.get_dummies(test_df[numeric_features + categorical_features],
                              columns=categorical_features, drop_first=True)
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)
train_encoded = train_encoded.fillna(train_encoded.median())
test_encoded = test_encoded.fillna(test_encoded.median())

X_train = train_encoded
y_train = train_df[target]
X_test = test_encoded
y_test = test_df[target]

print(f"Features: {X_train.shape[1]}")
print(f"\nKey advantage: No feature scaling needed for tree-based models!")

## Model 1: Decision Tree

In [ ]:
dt = DecisionTreeClassifier(
    max_depth=8, min_samples_split=20, min_samples_leaf=10,
    max_features='sqrt', class_weight='balanced', random_state=RANDOM_STATE
)
start = time.time()
dt.fit(X_train, y_train)
dt_time = time.time() - start

dt_pred = dt.predict(X_test)
dt_prob = dt.predict_proba(X_test)[:, 1]

print("=== Decision Tree Results ===")
print(f"Training time: {dt_time:.3f}s")
print(f"Accuracy:  {accuracy_score(y_test, dt_pred):.4f}")
print(f"Precision: {precision_score(y_test, dt_pred):.4f}")
print(f"Recall:    {recall_score(y_test, dt_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, dt_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, dt_prob):.4f}")

## Model 2: Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_split=10,
    min_samples_leaf=5, max_features='sqrt', class_weight='balanced',
    n_jobs=-1, random_state=RANDOM_STATE
)
start = time.time()
rf.fit(X_train, y_train)
rf_time = time.time() - start

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest Results ===")
print(f"Training time: {rf_time:.3f}s")
print(f"Accuracy:  {accuracy_score(y_test, rf_pred):.4f}")
print(f"Precision: {precision_score(y_test, rf_pred):.4f}")
print(f"Recall:    {recall_score(y_test, rf_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, rf_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, rf_prob):.4f}")

## Model 3: XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=150, learning_rate=0.1, max_depth=5,
    min_child_weight=3, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    use_label_encoder=False, eval_metric='logloss', random_state=RANDOM_STATE
)
start = time.time()
xgb.fit(X_train, y_train)
xgb_time = time.time() - start

xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)[:, 1]

print("=== XGBoost Results ===")
print(f"Training time: {xgb_time:.3f}s")
print(f"Accuracy:  {accuracy_score(y_test, xgb_pred):.4f}")
print(f"Precision: {precision_score(y_test, xgb_pred):.4f}")
print(f"Recall:    {recall_score(y_test, xgb_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, xgb_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, xgb_prob):.4f}")

## Side-by-Side Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'XGBoost'],
    'Accuracy': [accuracy_score(y_test, p) for p in [dt_pred, rf_pred, xgb_pred]],
    'Precision': [precision_score(y_test, p) for p in [dt_pred, rf_pred, xgb_pred]],
    'Recall': [recall_score(y_test, p) for p in [dt_pred, rf_pred, xgb_pred]],
    'F1 Score': [f1_score(y_test, p) for p in [dt_pred, rf_pred, xgb_pred]],
    'ROC-AUC': [roc_auc_score(y_test, p) for p in [dt_prob, rf_prob, xgb_prob]],
    'Train Time (s)': [dt_time, rf_time, xgb_time]
})
print("=" * 80)
print("TREE-BASED MODELS: HEAD-TO-HEAD COMPARISON")
print("=" * 80)
print(results.to_string(index=False))

In [ ]:
fig = go.Figure()
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
colors = ['#2ecc71', '#3498db', '#e74c3c']

for i, model in enumerate(results['Model']):
    fig.add_trace(go.Bar(name=model, x=metrics,
        y=[results.loc[i, m] for m in metrics], marker_color=colors[i]))

fig.update_layout(title='Tree-Based Models: Performance Comparison',
    yaxis_title='Score', barmode='group', height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

## Feature Importance
- All three models provide built-in feature importance scores
- Which student characteristics predict departure?

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Decision Tree': dt.feature_importances_,
    'Random Forest': rf.feature_importances_,
    'XGBoost': xgb.feature_importances_
})
importance_df = importance_df.sort_values('Random Forest', ascending=False).head(15)

fig = make_subplots(rows=1, cols=3, subplot_titles=('Decision Tree', 'Random Forest', 'XGBoost'))
for col, model in enumerate(['Decision Tree', 'Random Forest', 'XGBoost'], 1):
    sorted_df = importance_df.sort_values(model, ascending=True)
    fig.add_trace(go.Bar(y=sorted_df['Feature'], x=sorted_df[model],
        orientation='h', marker_color=colors[col-1], showlegend=False), row=1, col=col)

fig.update_layout(height=500, title_text='Top 15 Features by Importance')
fig.show()

## Visualizing the Decision Tree
- Biggest advantage of Decision Trees: you can **see** the model
- Perfect for showing advisors and administrators

In [ ]:
import matplotlib.pyplot as plt

dt_visual = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
dt_visual.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_visual, feature_names=X_train.columns, class_names=['Enrolled', 'Departed'],
          filled=True, rounded=True, fontsize=10, ax=ax)
plt.title('Decision Tree for Student Departure (max_depth=3)', fontsize=16)
plt.tight_layout()
plt.show()

print("\nThis tree can be shared directly with academic advisors!")

## Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models_cv = {
    'Decision Tree': DecisionTreeClassifier(max_depth=8, min_samples_split=20,
        min_samples_leaf=10, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12,
        min_samples_leaf=5, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, use_label_encoder=False,
        eval_metric='logloss', random_state=RANDOM_STATE)
}

print("5-Fold Cross-Validation Results (ROC-AUC):")
print("-" * 50)
for name, model in models_cv.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f"{name:20s}: {scores.mean():.4f} (+/- {scores.std():.4f})")

## Key Takeaways
- Same workflow for all three: `instantiate -> fit -> predict`
- No preprocessing/scaling needed
- All provide feature importance scores
- Decision Trees are visualizable for stakeholders
- **Next:** 3.3 Tuning Tree-Based Models